<!--<badge>--><a href="https://colab.research.google.com/github/JoeChen322/Fintech/blob/main/classification-ml-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->


# Investment Product Classification

This notebook implements a comprehensive machine learning pipeline to classify clients based on their investment profile and recommend suitable financial products. The workflow encompasses data cleaning, outlier detection, feature engineering, and model training using multiple classifiers.

## Objective

Build accurate binary classifiers for two investment targets:

- **Income Investment**: Investment in income-generating products
- **Accumulation Investment**: Investment in wealth accumulation/growth products

## Workflow

1. Load and explore client dataset
2. Clean and validate data
3. Detect and analyze outliers
4. Transform highly skewed features (Income, Wealth)
5. Engineer domain-specific features
6. Train and tune multiple classifiers
7. Benchmark and compare models


## Table of Contents

1. [Imports & Dependencies](#1-imports--dependencies)
2. [Configuration & Data Loading](#2-configuration--data-loading)
3. [Data Cleaning Functions](#3-data-cleaning-functions)
4. [Outlier Detection & Analysis](#4-outlier-detection--analysis)
5. [Data Transformation](#5-data-transformation)
6. [Feature Engineering](#6-feature-engineering)
7. [Model Training & Evaluation Framework](#7-model-training--evaluation-framework)
   - [7.1 ML Pipelines Setup](#71-ml-pipelines-setup)
8. [Model Training & Hyperparameter Tuning](#8-model-training--hyperparameter-tuning)
   - [8.1 SVM & Naive Bayes](#81-svm--naive-bayes)
   - [8.2 XGBoost](#82-xgboost)
   - [8.3 K-Nearest Neighbors (KNN)](#83-k-nearest-neighbors-knn)
   - [8.4 Random Forest](#84-random-forest)
9. [Model Benchmark & Comparison](#9-model-benchmark--comparison)


# Imports & Dependencies


In [5]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

#import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Other utilities
import xgboost as xgb

# Deep Learning - TabNet
from scipy import stats
from scipy.stats import chi2_contingency, normaltest, shapiro, skew
from sklearn.base import clone
from sklearn.ensemble import (
    IsolationForest,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)

# Feature selection
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    mutual_info_classif,
)
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Model selection and evaluation
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Pipeline
from sklearn.pipeline import Pipeline

# Data preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Classifiers
from sklearn.svm import SVC
from tabulate import tabulate

In [6]:
# Load data from local file instead of Google Drive
import os

# Define the local file path (relative to the notebook location)
current_dir = os.path.dirname(os.path.abspath("C:/Users/qzc20/Desktop/finech/Fintech"))
file_path = os.path.join(current_dir, "Dataset2_Needs.xls")

# If file is not found in the current directory, try the project root
if not os.path.exists(file_path):
    file_path = "Dataset2_Needs.xls"
    
print(f"Loading data from: {file_path}")


Loading data from: Dataset2_Needs.xls


# Configuration & Data Loading


In [7]:
needs_df = pd.read_excel(file_path, sheet_name="Needs")
products_df = pd.read_excel(file_path, sheet_name="Products")

# The metadata retrieved is that relating to the needs_df database
metadata_df = pd.read_excel(file_path, sheet_name="Metadata", nrows=11)

# We delete the first line, which doesn't serve much purpose
metadata_df = metadata_df[metadata_df["Metadata"] != "Clients"]

# We remove any potentially unnecessary spaces at the beginning and end of each column name.
# (An unwanted space was present in "income ")
needs_df.columns = needs_df.columns.str.strip()
products_df.columns = products_df.columns.str.strip()
metadata_df.columns = metadata_df.columns.str.strip()

# List of categorical columns in the dataset:
CATEGORICAL_COLS = ["Gender"]

# List of numeric columns in the dataset:
NUMERIC_COLS = [
    "Age",
    "FamilyMembers",
    "FinancialEducation",
    "RiskPropensity",
    "Income",
    "Wealth",
]

# List of target columns in the dataset:
TARGET_COLS = ["IncomeInvestment", "AccumulationInvestment"]

# Value mapping: introduces dictionaries to convert numeric codes into readable labels
VALUE_MAPS: Dict[str, Dict[int, str]] = {
    "Gender": {0: "Male", 1: "Female"},
    "AccumulationInvestment": {0: "Low propensity", 1: "High propensity"},
    "IncomeInvestment": {0: "Low propensity", 1: "High propensity"},
}


# Data Cleaning Functions


Define data validation and preprocessing functions.


In [8]:
# Dataclass used to store the cleaned and prepared dataset
@dataclass
class PreparedFeatures:
    # Final prepared dataset of features after preprocessing
    X: pd.DataFrame

    # Metadata associated with this dataset (feature descriptions, etc.)
    features_metadata_df: pd.DataFrame

    # List of numerical columns kept in the dataset
    numeric_cols: List[str]

    # List of categorical columns kept in the dataset
    categorical_cols: List[str]

    # Columns that have zero variance
    removed_zero_variance_cols: List[str]

    # Columns with very low variance
    near_zero_variance_cols: List[str]

    # Audit log describing the preprocessing steps that were applied
    audit_lines: List[str]


@dataclass
class PreparedTargets:
    # Final prepared dataset of targets values after preprocessing
    Y: pd.DataFrame

    # Metadata associated with this dataset (feature descriptions, etc.)
    targets_metadata_df: pd.DataFrame

    # List of indexes retained in the dataset after removing rows with missing IncomeInvestment and AccumulationInvest values
    kept_index: pd.Index

    # number of lines with NaN value
    dropped_rows_missing_target: int

    # Audit log describing the preprocessing steps that were applied
    audit_lines: List[str]


@dataclass
class OutlierDetectionResult:
    # Dataset of features after potentially removing the outliers
    X_inliers: pd.DataFrame

    # Dataset of target values after potentially removing the outliers
    Y_inliers: pd.DataFrame

    # Column associating 1 if the point is considered normal, -1 otherwise
    full_outlier_labels: pd.Series

    # Anomaly score for each observation
    full_anomaly_scores: pd.Series

    # Number of outliers detected
    outlier_count: int

    # Porportion of outliers
    outlier_rate: float

    # Audit log describing the preprocessing steps that were applied
    audit_lines: List[str]


The following code allows you to create two new datasets, one containing all the features and customer data, the other containing the target variables


In [9]:
def split_features_and_targets(
    data: pd.DataFrame, target_cols: List[str] = TARGET_COLS
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    Y = data[target_cols].copy()

    drop_cols = target_cols.copy()
    X = data.drop(columns=drop_cols, errors="ignore").copy()

    return X, Y


In [10]:
X, Y = split_features_and_targets(data=needs_df)

targets_metadata_df = metadata_df[
    metadata_df["Metadata"].isin(TARGET_COLS)
].reset_index(drop=True)

features_metadata_df = metadata_df[
    ~metadata_df["Metadata"].isin(TARGET_COLS)
].reset_index(drop=True)


In [11]:
# Detect different forms of missing values and replaces them with np.nan
def normalize_missing_tokens(df: pd.DataFrame) -> pd.DataFrame:
    tokens = {"", " ", "NA", "N/A", "na", "n/a", "null", "None", "none", "-", "--"}
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].replace(list(tokens), np.nan)
    return out


# Detect whether a feature has near-zero variance
def detect_near_zero_variance(series: pd.Series) -> bool:
    s = series.dropna()
    counts = s.value_counts()

    # Compute the ratio between the most frequent and second most frequent values
    freq_ratio = counts.iloc[0] / max(counts.iloc[1], 1)

    # Compute the ratio between the number of different values and the total number of observations used to measure the diversity of values in this feature
    pct_unique = s.nunique() / len(s)
    return (freq_ratio >= 20) and (pct_unique <= 0.10)


def clean_and_audit_data(
    data: pd.DataFrame,
    features_metadata_df: pd.DataFrame,
    categorical_cols: List[str] = CATEGORICAL_COLS,
    numeric_cols: List[str] = NUMERIC_COLS,
    value_maps: Dict[str, Dict[int, str]] = VALUE_MAPS,
) -> PreparedFeatures:
    # Initialize an audit log to keep track of all cleaning steps
    audit_lines: List[str] = []

    # Create a working copy of the dataset and standardize missing-value tokens
    df = normalize_missing_tokens(data.copy())

    # Start audit report
    audit_lines.append("DATA AUDIT")
    audit_lines.append("=" * 80)
    audit_lines.append(f"Initial X shape (dataset of features): {df.shape}")

    # Remove fully duplicated rows
    exact_dupes = int(df.duplicated().sum())
    if exact_dupes > 0:
        df = df.drop_duplicates().copy()
    audit_lines.append(f"Exact duplicate rows removed: {exact_dupes}")

    # Remove duplicated IDs, keeping only the first occurrence
    duplicate_ids = int(df["ID"].duplicated().sum())
    if duplicate_ids > 0:
        df = df.drop_duplicates(subset=["ID"], keep="first").copy()
    audit_lines.append(f"Duplicate IDs removed: {duplicate_ids}")

    # Delete the columns 'ID' that doesn't contains any usefull information for the clustering
    df = df.drop(columns=["ID"])

    # Identify categorical columns
    # Identify numeric columns by excluding ID and categorical columns
    categorical_cols = [c for c in categorical_cols if c in df.columns]
    numeric_cols = [c for c in numeric_cols if c in df.columns]

    # Convert columns to numeric (for instance "2" to 2). Invalid values become NaN
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Validate categorical codes against allowed values defined in VALUE_MAPS
    # All invalid values become: NaN
    audit_lines.append("Invalid categorical codes replaced with missing values:")
    for col in categorical_cols:
        allowed = set(value_maps[col].keys()) if col in value_maps else None
        invalid_mask = ~df[col].isin(allowed) & df[col].notna()
        invalid_count = int(invalid_mask.sum())
        if invalid_count > 0:
            df.loc[invalid_mask, col] = np.nan
        audit_lines.append(f"  - {col}: {invalid_count}")

    # Record missing values before imputation
    audit_lines.append("Missing values before imputation:")
    for col, val in df.isna().sum().items():
        audit_lines.append(f"  - {col}: {int(val)}")

    # Impute missing numeric values with the median
    for col in numeric_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    # Impute missing categorical values with the most frquence value taken
    # If this value doesn't exist, it uses a fallback value defined from the allowed categories (VALUE_MAPS)
    # Finaly convert the entire column to ensure a consistent format usable by algorithms
    for col in categorical_cols:
        if df[col].isna().any():
            mode_value = df[col].mode(dropna=True)
            fallback = sorted(value_maps[col].keys())[0]
            fill_value = int(mode_value.iloc[0]) if not mode_value.empty else fallback
            df[col] = df[col].fillna(fill_value)

        df[col] = df[col].astype(int)

    # Record missing values after imputation
    audit_lines.append("Missing values after imputation:")
    for col, val in df.isna().sum().items():
        audit_lines.append(f"  - {col}: {int(val)}")

    # Detect zero-variance and near-zero-variance features
    removed_zero_variance_cols: List[str] = []
    near_zero_variance_cols: List[str] = []

    # Iterate through the columns
    # Save them that contained only a single value (and therefore having a variance of 0) in the list removed_zero_variance_cols
    # Save them that detecte as "near zero variance" in the list near_zero_variance_cols
    for col in df.columns:
        if df[col].nunique(dropna=False) <= 1:
            removed_zero_variance_cols.append(col)
        elif detect_near_zero_variance(df[col]):
            near_zero_variance_cols.append(col)

    # Remove zero-variance columns
    if removed_zero_variance_cols:
        df = df.drop(columns=removed_zero_variance_cols).copy()

    # Remove near-zero-variance columns
    if near_zero_variance_cols:
        df = df.drop(columns=near_zero_variance_cols).copy()

    audit_lines.append(
        "Zero-variance columns removed: "
        + (
            ", ".join(removed_zero_variance_cols)
            if removed_zero_variance_cols
            else "none"
        )
    )
    audit_lines.append(
        "Near-zero-variance columns removed: "
        + (", ".join(near_zero_variance_cols) if near_zero_variance_cols else "none")
    )

    # Update column lists after removing zero-variance and near-zero-variance columns
    categorical_cols = [
        c for c in categorical_cols if c not in removed_zero_variance_cols
    ]
    numeric_cols = [c for c in numeric_cols if c not in removed_zero_variance_cols]

    # Compute absolute correlation matrix for numeric features
    if numeric_cols:
        corr = df[numeric_cols].corr().abs()
        high_corr = []

        for i in range(len(corr.columns)):
            for j in range(i + 1, len(corr.columns)):
                if corr.iloc[i, j] >= 0.85:
                    high_corr.append(
                        (corr.index[i], corr.columns[j], float(corr.iloc[i, j]))
                    )

        if high_corr:
            audit_lines.append(
                "High-correlation numeric pairs flagged (not auto-removed):"
            )
            for a, b, r in high_corr:
                audit_lines.append(f"  - {a} vs {b}: {r:.3f}")
        else:
            audit_lines.append(
                "High-correlation numeric pairs flagged: none above 0.85"
            )
    else:
        audit_lines.append(
            "High-correlation numeric pairs flagged: no numeric columns available"
        )

    audit_lines.append(f"Final shape after cleaning: {df.shape}")

    return PreparedFeatures(
        X=df,
        features_metadata_df=features_metadata_df,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        removed_zero_variance_cols=removed_zero_variance_cols,
        near_zero_variance_cols=near_zero_variance_cols,
        audit_lines=audit_lines,
    )


Binary target processing :


In [12]:
def prepare_binary_targets(
    data: pd.DataFrame,
    targets_metadata_df: pd.DataFrame,
    target_cols: List[str] = TARGET_COLS,
    value_maps: Dict[str, Dict[int, str]] = VALUE_MAPS,
) -> PreparedTargets:
    # Initialize an audit log to keep track of all cleaning steps
    audit_lines: List[str] = []

    # Create a working copy of the dataset and standardize missing-value tokens
    df = normalize_missing_tokens(data.copy())

    # Start audit report
    audit_lines.append("DATA AUDIT")
    audit_lines.append("=" * 80)
    audit_lines.append(f"Initial Y shape (dataset of targets): {df.shape}")

    audit_lines.append(f"Invalid target codes replaced with missing values")
    for col in df.columns:
        # Convert columns to numeric (for instance "2" to 2). Invalid values become NaN
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Validate values against allowed values defined in VALUE_MAPS
        # All invalid values become: NaN
        allowed = set(value_maps[col].keys())
        invalid_mask = ~df[col].isin(allowed) & df[col].notna()
        invalid_count = int(invalid_mask.sum())
        if invalid_count > 0:
            df.loc[invalid_mask, col] = np.nan

        audit_lines.append(f"  - {col}: {invalid_count}")

    audit_lines.append("Missing values in targets before filtering:")
    for col, val in df.isna().sum().items():
        audit_lines.append(f"  - {col}: {int(val)}")

    # Remove the lines where a target is missing
    mask_complete_targets = df.notna().all(axis=1)

    # number of lines with NaN value
    dropped_rows = int((~mask_complete_targets).sum())

    df = df.loc[mask_complete_targets].copy()

    # List of retained indexes
    kept_index = df.index

    # Conversion finale en int
    for col in df.columns:
        df[col] = df[col].astype(int)

    audit_lines.append(f"Rows dropped because of missing target(s): {dropped_rows}")
    audit_lines.append(f"Final y shape after filtering: {df.shape}")

    return PreparedTargets(
        Y=df,
        targets_metadata_df=targets_metadata_df,
        kept_index=kept_index,
        dropped_rows_missing_target=dropped_rows,
        audit_lines=audit_lines,
    )


In [13]:
prepared_Y = prepare_binary_targets(data=Y, targets_metadata_df=targets_metadata_df)

# We filter raw X based on the rows retained in Y
X_filtered = X.loc[prepared_Y.kept_index].copy()

prepared_X = clean_and_audit_data(
    data=X_filtered, features_metadata_df=features_metadata_df
)

# Final security: realign Y on the lines actually preserved in X
X_clean = prepared_X.X
Y_clean = prepared_Y.Y.loc[X_clean.index].copy()


In [14]:
X_clean.head()


,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth
0,60,0,2,0.228685,0.233355,68.181525,53.260067
1,78,0,2,0.358916,0.170911,21.807595,135.550048
2,33,1,2,0.317515,0.249703,23.252747,66.303678
3,69,1,4,0.767685,0.654597,166.189034,404.997689
4,58,0,3,0.429719,0.349039,21.186723,58.911930


In [15]:
Y_clean.head()


,IncomeInvestment,AccumulationInvestment
0,0,1
1,1,0
2,0,1
3,1,1
4,0,0


# Outlier Detection & Analysis


In [16]:
def detect_outliers_isolation_forest(
    X: pd.DataFrame,
    Y: pd.DataFrame,
    numeric_cols: List[str] = NUMERIC_COLS,
    score_percentile: float = 5.0,
    n_estimators: int = 300,
    random_state: int = 42,
    remove_outliers: bool = False,
) -> OutlierDetectionResult:

    # Detect atypical observations in X with Isolation Forest, then possibly remove them from X and Y
    audit_lines: List[str] = []
    audit_lines.append("OUTLIER DETECTION WITH ISOLATION FOREST")
    audit_lines.append("=" * 80)
    audit_lines.append(f"Input X shape: {X.shape}")
    audit_lines.append(f"Score percentile threshold: {score_percentile}")
    audit_lines.append(f"n_estimators: {n_estimators}")

    # Standardize the numerical variables (excluding gender)
    X_scaled = X.copy()
    scaler = StandardScaler()
    X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    iso = IsolationForest(n_estimators=n_estimators, random_state=random_state)
    iso.fit(X_scaled)

    # Returns an anomaly score
    # The lower the score, the more suspicious the observation.
    scores = iso.decision_function(X_scaled)

    # Convert scores to pandas Series
    scores_series = pd.Series(scores, index=X.index, name="AnomalyScore")

    # Threshold based on the score distribution
    score_threshold = np.percentile(scores, score_percentile)

    # Build labels manually from the threshold
    # -1: atypical observation
    #  1: normal observation
    labels = np.where(scores <= score_threshold, -1, 1)
    labels_series = pd.Series(labels, index=X.index, name="OutlierLabel")

    outlier_count = int((labels_series == -1).sum())
    outlier_rate = outlier_count / len(X)

    audit_lines.append(f"Score threshold used: {score_threshold:.6f}")
    audit_lines.append(f"Outliers detected: {outlier_count}")
    audit_lines.append(f"Outlier rate: {outlier_rate:.2%}")

    if remove_outliers:
        inlier_mask = labels_series == 1
        # keep only the normal lines in X and Y
        X_inliers = X.loc[inlier_mask].copy()
        Y_inliers = Y.loc[inlier_mask].copy()
        audit_lines.append("Outliers removed from returned datasets: yes")
    else:
        X_inliers = X.copy()
        Y_inliers = Y.copy()
        audit_lines.append("Outliers removed from returned datasets: no")

    return OutlierDetectionResult(
        X_inliers=X,
        Y_inliers=Y,
        full_outlier_labels=labels_series,
        full_anomaly_scores=scores_series,
        outlier_count=outlier_count,
        outlier_rate=outlier_rate,
        audit_lines=audit_lines,
    )


In [17]:
outlier_result = detect_outliers_isolation_forest(X=X_clean, Y=Y_clean)


In [18]:
print("\n".join(outlier_result.audit_lines))

# Jeux finaux pour le modèle
X_final = outlier_result.X_inliers
Y_final = outlier_result.Y_inliers

print("\nFINAL DATA FOR MODEL")
print("=" * 80)
print(f"X_final shape: {X_final.shape}")
print(f"Y_final shape: {Y_final.shape}")

print("\nTarget distributions after filtering:")
for col in Y_final.columns:
    print(Y_final[col].value_counts(dropna=False).sort_index())

diagnostics_df = pd.DataFrame(
    {
        "OutlierLabel": outlier_result.full_outlier_labels,
        "AnomalyScore": outlier_result.full_anomaly_scores,
    }
)

print("\nFirst rows of outlier diagnostics:")
diagnostics_df.head(10)


OUTLIER DETECTION WITH ISOLATION FOREST
Input X shape: (5000, 7)
Score percentile threshold: 5.0
n_estimators: 300
Score threshold used: -0.082109
Outliers detected: 250
Outlier rate: 5.00%
Outliers removed from returned datasets: no

FINAL DATA FOR MODEL
X_final shape: (5000, 7)
Y_final shape: (5000, 2)

Target distributions after filtering:
IncomeInvestment
0    3082
1    1918
Name: count, dtype: int64
AccumulationInvestment
0    2434
1    2566
Name: count, dtype: int64

First rows of outlier diagnostics:


,OutlierLabel,AnomalyScore
0,1,0.069370
1,1,0.005807
2,1,0.022397
3,-1,-0.137063
4,1,0.083244
5,1,0.047337
6,1,0.094361
7,1,0.044447
8,1,-0.025458
9,1,0.022178


In [19]:
outliers_df = outlier_result.X_inliers[outlier_result.full_outlier_labels == -1].copy()
print("\nLIST OF ALL OUTLIERS FOUND")
outliers_df.head(10)


LIST OF ALL OUTLIERS FOUND


,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth
3,69,1,4,0.767685,0.654597,166.189034,404.997689
17,80,1,1,0.119173,0.098889,6.802567,4.054744
23,81,0,2,0.595024,0.340589,156.063265,462.458278
117,25,0,1,0.571516,0.745596,3.602805,41.655649
122,87,0,4,0.157120,0.060139,13.266355,350.469602
154,78,0,2,0.130559,0.628745,36.812458,655.002664
164,82,1,3,0.222326,0.154917,144.715628,567.784795
231,32,0,4,0.134606,0.163278,196.829694,64.088322
241,76,1,4,0.245409,0.336510,162.155229,338.146963
256,78,1,1,0.156416,0.102797,170.600508,242.172707


In [20]:
desc = outlier_result.X_inliers.describe().T
desc["skewness"] = outlier_result.X_inliers.skew()
desc["kurtosis"] = outlier_result.X_inliers.kurtosis()
desc["IQR"] = desc["75%"] - desc["25%"]
desc["CV"] = desc["std"] / desc["mean"]  # Coefficient of variation
print(desc.round(4).to_string())

                     count     mean       std      min      25%      50%       75%        max  skewness  kurtosis      IQR      CV
Age                 5000.0  55.2534   11.9717  18.0000  47.0000  55.0000   63.0000    97.0000    0.0586   -0.0251  16.0000  0.2167
Gender              5000.0   0.4920    0.5000   0.0000   0.0000   0.0000    1.0000     1.0000    0.0320   -1.9998   1.0000  1.0162
FamilyMembers       5000.0   2.5106    0.7618   1.0000   2.0000   3.0000    3.0000     5.0000    0.0647   -0.2084   1.0000  0.3035
FinancialEducation  5000.0   0.4191    0.1514   0.0361   0.3081   0.4167    0.5234     0.9029    0.1485   -0.4221   0.2153  0.3612
RiskPropensity      5000.0   0.3627    0.1511   0.0248   0.2464   0.3545    0.4671     0.8827    0.3261   -0.3494   0.2207  0.4167
Income              5000.0  62.9943   44.3598   1.5378  30.5965  53.3994   84.1223   365.3234    1.3773    2.8560  53.5258  0.7042
Wealth              5000.0  93.8063  105.4710   1.0574  38.3111  66.0706  114.8248 

In [21]:
# Apply power transformations to normalize skewed financial variables
# Wealth: Power = 0.124, Income: Power = 0.313
outlier_result.X_inliers["Wealth_pow"] = np.power(
    outlier_result.X_inliers["Wealth"], 0.124
)
outlier_result.X_inliers["Income_pow"] = np.power(
    outlier_result.X_inliers["Income"], 0.313
)

"""
# Apply MinMaxScaler to numerical variables
scaler = MinMaxScaler()
vars_to_normalize = ['Age', 'RiskPropensity', 'Wealth_pow', 'Income_pow']
outlier_result.X_inliers[vars_to_normalize] = scaler.fit_transform(outlier_result.X_inliers[vars_to_normalize])
"""
# Get all numeric columns including transformed ones (and not transformed, for comparison)
numeric_cols = outlier_result.X_inliers.select_dtypes(
    include=["float64", "int64"]
).columns

# Feature Engineering


Create domain-specific features including ratios, interactions, and life-stage indicators.


1. **Household-Adjusted Metrics** (Ratios)

These features normalize raw financial data based on the family structure to better reflect "actual" financial capacity.

**IncomePerFamilyMember&WealthPerFamilyMember**: Instead of looking at total income, these measure the disposable resources available per person. A high income divided by many members indicates less investment "surplus" than the same income for a single individual.

**WealthIncomeRatio**: This measures financial stability. A high ratio suggests the household has significant accumulated assets relative to their earnings, often identifying "Wealthy" profiles who might prioritize capital preservation over aggressive growth.

2. **Interaction Features**

Interactions capture how two variables behave differently when combined, allowing models (especially linear ones) to see non-linear patterns.

**RiskEducationInteraction**: This represents the "Informed Risk" profile. It distinguishes between someone who takes risks due to lack of knowledge and someone who takes calculated risks based on financial literacy.

**RiskWealthInteraction**: This captures the "Capacity for Loss." High risk propensity combined with high wealth indicates a client who can afford aggressive investment strategies without jeopardizing their basic needs.

**AgeRiskInteraction**: This models the "Time Horizon." Younger individuals with high risk tolerance have time to recover from market drops, whereas older individuals with high risk tolerance still face a shorter investment window, changing their optimal portfolio.

3. **Non-linearity and Life-Stage Flags**

These features help the model understand that human behavior doesn't change linearly as we get older.

**AgeSquared**: This accounts for the "Life-Cycle Hypothesis." Wealth and investment activity typically follow a U-shaped or parabolic curve—increasing during peak working years and decreasing during retirement.

**Life-Stage Flags (Age_Under35, 35_54, etc.)**: These transform age into categorical milestones. Instead of treating age as just a number, these flags tell the model: "This person is a 'Young Professional' or a 'Retiree'." Each stage has distinct financial goals (e.g., buying a first home vs. estate planning).


In [22]:
final_X = outlier_result.X_inliers
final_Y = outlier_result.Y_inliers


def prepare_features(df):
    X = df.copy()

    # Ratios / household-adjusted metrics
    X["IncomePerFamilyMember"] = X["Income"] / (X["FamilyMembers"] + 1.0)
    X["WealthPerFamilyMember"] = X["Wealth"] / (X["FamilyMembers"] + 1.0)
    X["WealthIncomeRatio"] = X["Wealth"] / (X["Income"] + 1.0)

    # Interactions
    X["RiskEducationInteraction"] = X["RiskPropensity"] * X["FinancialEducation"]
    X["RiskWealthInteraction"] = X["RiskPropensity"] * X["Wealth_pow"]
    X["AgeRiskInteraction"] = X["Age"] * X["RiskPropensity"]

    # Nonlinearity
    X["AgeSquared"] = X["Age"] ** 2

    # Life-stage flags with fixed cutoffs (not learned from whole-dataset quantiles)
    X["Age_Under35"] = (X["Age"] < 35).astype(int)
    X["Age_35_54"] = ((X["Age"] >= 35) & (X["Age"] < 55)).astype(int)
    X["Age_55_69"] = ((X["Age"] >= 55) & (X["Age"] < 70)).astype(int)
    X["Age_70plus"] = (X["Age"] >= 70).astype(int)

    # Select features for modeling
    features_base = [
        "Age",
        "Gender",
        "FamilyMembers",
        "FinancialEducation",
        "RiskPropensity",
        "Wealth_pow",
        "Income_pow",
    ]

    features_engineered = [
        "Age",
        "Gender",
        "FamilyMembers",
        "FinancialEducation",
        "RiskPropensity",
        "IncomePerFamilyMember",
        "WealthPerFamilyMember",
        "WealthIncomeRatio",
        "RiskEducationInteraction",
        "RiskWealthInteraction",
        "AgeRiskInteraction",
        "AgeSquared",
        "Age_Under35",
        "Age_35_54",
        "Age_55_69",
        "Age_70plus",
    ]

    # Define categorical and numerical features
    categorical_features = ["Gender", "FinancialEducation"]

    # Normalize numerical features only
    numerical_features = [
        col for col in features_base if col not in categorical_features
    ]
    scaler = StandardScaler()
    X_base_scaled = X[features_base].copy()
    X_base_scaled[numerical_features] = scaler.fit_transform(X[numerical_features])
    X_base = X_base_scaled[features_base]  # Ensure column order matches features_base

    # For engineered features, also scale only numerical features
    engineered_numerical = [
        col for col in features_engineered if col not in categorical_features
    ]
    scaler_eng = StandardScaler()
    X_eng_scaled = X[features_engineered].copy()
    X_eng_scaled[engineered_numerical] = scaler_eng.fit_transform(
        X[engineered_numerical]
    )
    X_engineered = X_eng_scaled[
        features_engineered
    ]  # Ensure column order matches features_engineered

    return X_base, X_engineered


# Prepare features
X_base, X_engineered = prepare_features(final_X)
y_income = final_Y["IncomeInvestment"]
y_accum = final_Y["AccumulationInvestment"]


# Model Training & Evaluation Framework


Define helper functions for model training, evaluation, and results visualization.


In [23]:
def split_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    return X_train, X_test, y_train, y_test


def train_evaluate_model(X_train, y_train, X_test, y_test, model, k_folds=5):
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    cv_metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold)
        y_val_pred = model.predict(X_val_fold)

        cv_metrics["accuracy"].append(accuracy_score(y_val_fold, y_val_pred))
        cv_metrics["precision"].append(precision_score(y_val_fold, y_val_pred))
        cv_metrics["recall"].append(recall_score(y_val_fold, y_val_pred))
        cv_metrics["f1"].append(f1_score(y_val_fold, y_val_pred))

    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)

    return {
        "cv_metrics": {
            metric: {"mean": np.mean(scores), "std": np.std(scores)}
            for metric, scores in cv_metrics.items()
        },
        "test_metrics": {
            "accuracy": accuracy_score(y_test, y_test_pred),
            "precision": precision_score(y_test, y_test_pred),
            "recall": recall_score(y_test, y_test_pred),
            "f1": f1_score(y_test, y_test_pred),
        },
    }


def display_results_table(results_dict, model_name, feature_type):
    cv_data = {
        "Metric": ["Accuracy", "Precision", "Recall", "F1"],
        "CV Mean": [
            results_dict["cv_metrics"]["accuracy"]["mean"],
            results_dict["cv_metrics"]["precision"]["mean"],
            results_dict["cv_metrics"]["recall"]["mean"],
            results_dict["cv_metrics"]["f1"]["mean"],
        ],
        "CV Std": [
            results_dict["cv_metrics"]["accuracy"]["std"],
            results_dict["cv_metrics"]["precision"]["std"],
            results_dict["cv_metrics"]["recall"]["std"],
            results_dict["cv_metrics"]["f1"]["std"],
        ],
        "Test Set": [
            results_dict["test_metrics"]["accuracy"],
            results_dict["test_metrics"]["precision"],
            results_dict["test_metrics"]["recall"],
            results_dict["test_metrics"]["f1"],
        ],
    }

    df = pd.DataFrame(cv_data)
    df = df.round(3)

    print(f"\n{model_name} - {feature_type}")
    print("=" * 60)
    print(tabulate(df, headers="keys", tablefmt="pretty"))


# Model Training & Hyperparameter Tuning

with our personalized features using scikit-learn Pipelines for reproducible and maintainable ML workflows.


Train and optimize multiple classification models using cross-validation.


## ML Pipelines Setup

We use scikit-learn Pipelines to encapsulate preprocessing + model training. This ensures consistent data handling, simplifies hyperparameter tuning with GridSearchCV/RandomizedSearchCV, and maintains code clarity.


In [24]:
def create_pipeline(classifier, scaler=True, feature_selector=None, n_features="auto"):
    """
    Create a pipeline with optional scaler, feature selector, and a classifier.

    Args:
        classifier: The estimator (SVC, RandomForest, KNN, etc.)
        scaler: Whether to include StandardScaler (default True for robustness)
        feature_selector: Type of feature selection ("kbest", "mutual_info", None)
        n_features: Number of features to select ("auto" for sqrt(n_features), or int)

    Returns:
        Pipeline object with (scaler, feature_selector, classifier) steps
    """
    steps = []

    # Step 1: Scaling
    if scaler:
        steps.append(("scaler", StandardScaler()))

    # Step 2: Feature Selection
    if feature_selector == "kbest":
        steps.append(("feature_selector", SelectKBest(f_classif, k=n_features)))
    elif feature_selector == "mutual_info":
        steps.append(
            ("feature_selector", SelectKBest(mutual_info_classif, k=n_features))
        )
    # If None, no feature selection

    # Step 3: Classifier
    steps.append(("classifier", classifier))

    return Pipeline(steps)


## Hyperparameter Tuning


## Ensemble Methods

### Voting Classifier

Combines predictions from multiple independent base learners using a voting strategy:

- **Soft Voting**: Averages predicted probabilities across all classifiers, then assigns the class with the highest average probability
  - Better calibrated predictions
  - Leverages confidence scores from individual models
- **Hard Voting**: Uses majority voting on predicted class labels
  - More robust to individual model errors
  - Simple and interpretable
- **Base Learners**: SVM, XGBoost, and Random Forest
- **Advantage**: Simple and effective ensemble that captures diverse learning patterns

### Stacking Classifier

Uses a meta-learner to optimally combine predictions from multiple base learners:

- **Base Learners**: SVM, XGBoost, K-Nearest Neighbors, and Random Forest
- **Meta-Learner**: Logistic Regression (learns optimal weights for combining base predictions)
- **Cross-Validation**: Uses 5-fold CV on training data to generate meta-features
- **Advantage**: More sophisticated approach that learns how to best combine diverse models


Compare performance across all models and feature sets.


In [25]:
def _cv_summary(estimator, X_train, y_train, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1"],
        n_jobs=-1,
        error_score="raise",
    )
    return {
        "CV Accuracy (mean)": float(np.mean(scores["test_accuracy"])),
        "CV Accuracy (std)": float(np.std(scores["test_accuracy"])),
        "CV Precision (mean)": float(np.mean(scores["test_precision"])),
        "CV Precision (std)": float(np.std(scores["test_precision"])),
        "CV Recall (mean)": float(np.mean(scores["test_recall"])),
        "CV Recall (std)": float(np.std(scores["test_recall"])),
        "CV F1 (mean)": float(np.mean(scores["test_f1"])),
        "CV F1 (std)": float(np.std(scores["test_f1"])),
    }


def _test_summary(estimator, X_train, y_train, X_test, y_test):
    est = clone(estimator)
    est.fit(X_train, y_train)
    y_pred = est.predict(X_test)
    return {
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
    }

In [26]:
rows = []

feature_sets = [
    ("Base", X_base),
    ("Engineered", X_engineered),
]

for target_name, y in [
    ("Income Investment", y_income),
    ("Accumulation Investment", y_accum),
]:
    for feat_label, X_data in feature_sets:
        X_train, X_test, y_train, y_test = split_data(X_data, y)

        # Baseline models using Pipelines for reproducibility
        models = {
            "SVM-baseline": create_pipeline(SVC(random_state=42), scaler=True),
            "NaiveBayes-baseline": create_pipeline(GaussianNB(), scaler=True),
            "KNN-baseline": create_pipeline(KNeighborsClassifier(), scaler=True),
            "RandomForest-baseline": create_pipeline(
                RandomForestClassifier(
                    n_estimators=50,
                    max_depth=10,
                    min_samples_split=20,
                    min_samples_leaf=10,
                    n_jobs=-1,
                    random_state=42
                ),
                scaler=False,
            ),
            "XGBoost-baseline": create_pipeline(
                xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
                scaler=False,  # Tree-based model, no scaling needed
            ),
        }

        # Voting Classifier: Combines predictions from multiple base learners
        # Uses soft voting to average predicted probabilities
        models["Voting (soft)"] = create_pipeline(
            VotingClassifier(
                estimators=[
                    ("knn", KNeighborsClassifier(n_neighbors=3, n_jobs=-1)),
                    ("rf", RandomForestClassifier(
                        n_estimators=40,
                        max_depth=10,
                        min_samples_split=20,
                        min_samples_leaf=10,
                        n_jobs=-1,
                        random_state=42
                    )),
                ],
                voting="soft",
            ),
            scaler=True,
        )

        # Hard Voting Classifier: Takes majority vote on class labels
        models["Voting (hard)"] = create_pipeline(
            VotingClassifier(
                estimators=[
                    ("knn", KNeighborsClassifier(n_neighbors=3, n_jobs=-1)),
                    ("rf", RandomForestClassifier(
                        n_estimators=50,
                        max_depth=10,
                        min_samples_split=20,
                        min_samples_leaf=10,
                        n_jobs=-1,
                        random_state=42
                    )),
                ],
                voting="hard",
            ),
            scaler=True,
        )

        # Stacking Classifier: Uses a meta-learner to combine base learner predictions
        # Meta-learner: Logistic Regression
        stacking_clf = StackingClassifier(
            estimators=[
                ("knn", KNeighborsClassifier(n_neighbors=3, n_jobs=-1)),
                ("rf", RandomForestClassifier(
                    n_estimators=50,
                    max_depth=10,
                    min_samples_split=20,
                    min_samples_leaf=10,
                    n_jobs=-1,
                    random_state=42
                )),
            ],
            final_estimator=LogisticRegression(),
            cv=3,  # Internal cross-validation to train the meta-learner
            stack_method="predict_proba",  # Passes probabilities to the meta-learner
        )
        models["Stacking (LogReg)"] = create_pipeline(stacking_clf, scaler=True)

        for model_name, model in models.items():
            base = {
                "Target": target_name,
                "Feature set": feat_label,
                "Model": model_name,
            }
            cv_part = _cv_summary(clone(model), X_train, y_train)
            test_part = _test_summary(model, X_train, y_train, X_test, y_test)
            rows.append({**base, **cv_part, **test_part})

In [27]:
bench_df = pd.DataFrame(rows)
bench_df = bench_df.sort_values(
    ["Target", "Feature set", "Test F1"], ascending=[True, True, False]
)
display(bench_df.round(4))

,Target,Feature set,Model,CV Accuracy (mean),CV Accuracy (std),CV Precision (mean),CV Precision (std),CV Recall (mean),CV Recall (std),CV F1 (mean),CV F1 (std),Test Accuracy,Test Precision,Test Recall,Test F1
19,Accumulation Investment,Base,RandomForest-baseline,0.7773,0.0143,0.8114,0.0159,0.7375,0.0147,0.7727,0.0145,0.775,0.7915,0.7622,0.7766
23,Accumulation Investment,Base,Stacking (LogReg),0.7750,0.0145,0.8017,0.0139,0.7462,0.0180,0.7729,0.0153,0.768,0.7771,0.7680,0.7725
20,Accumulation Investment,Base,XGBoost-baseline,0.7865,0.0137,0.8195,0.0137,0.7492,0.0218,0.7826,0.0154,0.768,0.7827,0.7583,0.7703
16,Accumulation Investment,Base,SVM-baseline,0.7537,0.0192,0.7866,0.0206,0.7141,0.0223,0.7485,0.0199,0.743,0.7591,0.7310,0.7448
21,Accumulation Investment,Base,Voting (soft),0.7350,0.0167,0.7544,0.0180,0.7175,0.0207,0.7354,0.0169,0.720,0.7316,0.7173,0.7244
22,Accumulation Investment,Base,Voting (hard),0.7410,0.0153,0.8524,0.0254,0.5996,0.0149,0.7039,0.0165,0.733,0.8187,0.6160,0.7030
18,Accumulation Investment,Base,KNN-baseline,0.6915,0.0108,0.7103,0.0107,0.6737,0.0152,0.6915,0.0117,0.683,0.6937,0.6842,0.6889
17,Accumulation Investment,Base,NaiveBayes-baseline,0.6070,0.0113,0.6160,0.0158,0.6259,0.0260,0.6203,0.0095,0.609,0.6130,0.6452,0.6287
27,Accumulation Investment,Engineered,RandomForest-baseline,0.8070,0.0096,0.8590,0.0109,0.7467,0.0203,0.7987,0.0118,0.803,0.8450,0.7544,0.7971
31,Accumulation Investment,Engineered,Stacking (LogReg),0.8050,0.0059,0.8458,0.0139,0.7589,0.0190,0.7997,0.0074,0.795,0.8222,0.7661,0.7931


# Optuna Hyperparameter Tuning for Random Forest

Use Optuna to optimize Random Forest hyperparameters with F1 score as the objective metric.

In [ ]:
# First, install optuna if not already installed
import subprocess
import sys

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
    import optuna

print("Optuna version:", optuna.__version__)

In [ ]:
def create_optuna_rf_optimizer(X_train, y_train, X_test, y_test, n_trials=100, cv_folds=5):
    """
    Create and run an Optuna study to optimize Random Forest hyperparameters.
    
    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_trials: Number of trials for Optuna to run
        cv_folds: Number of cross-validation folds
    
    Returns:
        best_params: Dictionary of best hyperparameters
        study: Optuna study object
        best_trial: Best trial object
    """
    
    def objective(trial):
        # Define hyperparameter search space
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 10, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        }
        
        # Create model with suggested parameters
        rf_model = RandomForestClassifier(
            **params,
            n_jobs=-1,
            random_state=42
        )
        
        # Use cross-validation to evaluate
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        accuracy_scores = []
        
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            rf_model.fit(X_train_fold, y_train_fold)
            y_pred = rf_model.predict(X_val_fold)
            accuracy_scores.append(accuracy_score(y_val_fold, y_pred))
        
        return np.mean(accuracy_scores)
    
    # Create and run Optuna study
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner()
    )
    
    # Optimize with timeout (in seconds)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    best_trial = study.best_trial
    best_params = best_trial.params
    
    return best_params, study, best_trial


# Example: Optimize for Income Investment with Engineered Features
print("\n" + "="*80)
print("OPTUNA HYPERPARAMETER TUNING FOR RANDOM FOREST - Income Investment (Engineered Features)")
print("="*80)

# First prepare the data (same as before)
X_train_eng, X_test_eng, y_train_income, y_test_income = split_data(X_engineered, y_income)

# Run Optuna optimization
best_params, study, best_trial = create_optuna_rf_optimizer(
    X_train_eng, y_train_income, X_test_eng, y_test_income,
    n_trials=100,
    cv_folds=5
)

print(f"\nOptimization completed in {len(study.trials)} trials")
print(f"Best Accuracy (CV): {best_trial.value:.4f}")
print("\nBest Hyperparameters:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Compare baseline vs optimized Random Forest performance
print("\n" + "="*80)
print("PERFORMANCE COMPARISON: BASELINE vs OPTUNA-OPTIMIZED")
print("="*80)

# Train baseline model (parameters from earlier)
baseline_rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42
)

# Train optimized model with best parameters
optimized_rf = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=42
)

# Train both models
baseline_rf.fit(X_train_eng, y_train_income)
optimized_rf.fit(X_train_eng, y_train_income)

# Get predictions and metrics for both
def get_metrics(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    return {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train Precision': precision_score(y_train, y_train_pred),
        'Test Precision': precision_score(y_test, y_test_pred),
        'Train Recall': recall_score(y_train, y_train_pred),
        'Test Recall': recall_score(y_test, y_test_pred),
        'Train F1': f1_score(y_train, y_train_pred),
        'Test F1': f1_score(y_test, y_test_pred),
    }

baseline_metrics = get_metrics(baseline_rf, X_train_eng, y_train_income, X_test_eng, y_test_income)
optimized_metrics = get_metrics(optimized_rf, X_train_eng, y_train_income, X_test_eng, y_test_income)

# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Optuna-Optimized': list(optimized_metrics.values()),
})

comparison_df['Improvement'] = comparison_df['Optuna-Optimized'] - comparison_df['Baseline']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Baseline'] * 100).round(2)

print("\n" + tabulate(comparison_df.round(4), headers='keys', tablefmt='pretty', showindex=False))

print(f"\n✓ Test Accuracy improvement: {comparison_df[comparison_df['Metric']=='Test Accuracy']['Improvement'].values[0]:.4f}")

In [ ]:
# Visualize Optuna optimization history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Optimization history (trial value over trials)
trials_df = study.trials_dataframe()
axes[0, 0].plot(range(len(trials_df)), trials_df['value'].values, 'b-', alpha=0.6, linewidth=1)
axes[0, 0].scatter(range(len(trials_df)), trials_df['value'].values, s=20, alpha=0.5)
axes[0, 0].axhline(y=best_trial.value, color='r', linestyle='--', label=f'Best: {best_trial.value:.4f}')
axes[0, 0].set_xlabel('Trial Number')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Optimization History')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Distribution of n_estimators in trials
n_est_values = [trial.params.get('n_estimators') for trial in study.trials]
axes[0, 1].hist(n_est_values, bins=20, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=best_params['n_estimators'], color='r', linestyle='--', linewidth=2, label='Best')
axes[0, 1].set_xlabel('n_estimators')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of n_estimators')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Accuracy vs max_depth
max_depth_values = [trial.params.get('max_depth') for trial in study.trials]
accuracy_values = [trial.value for trial in study.trials]
axes[1, 0].scatter(max_depth_values, accuracy_values, alpha=0.6, s=50)
axes[1, 0].scatter([best_params['max_depth']], [best_trial.value], color='r', s=200, marker='*', label='Best', zorder=5)
axes[1, 0].set_xlabel('max_depth')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Accuracy vs max_depth')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Accuracy vs min_samples_split
min_split_values = [trial.params.get('min_samples_split') for trial in study.trials]
axes[1, 1].scatter(min_split_values, accuracy_values, alpha=0.6, s=50, c='green')
axes[1, 1].scatter([best_params['min_samples_split']], [best_trial.value], color='r', s=200, marker='*', label='Best', zorder=5)
axes[1, 1].set_xlabel('min_samples_split')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Accuracy vs min_samples_split')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete!")

In [ ]:
# Optional: Optimize for both targets with a reusable function
def optimize_rf_for_all_targets(X_train_data, y_data, feature_label, n_trials=50):
    """
    Optimize Random Forest for all targets and return results.
    
    Args:
        X_train_data: Training features
        y_data: Dictionary with target names as keys and target Series as values
        feature_label: Label for feature set (e.g., "Base" or "Engineered")
        n_trials: Number of trials for each target
    
    Returns:
        results: Dictionary with optimization results for each target
    """
    results = {}
    
    for target_name, y in y_data.items():
        print(f"\n{'='*80}")
        print(f"Optimizing for {target_name} ({feature_label} Features) - {n_trials} trials")
        print(f"{'='*80}")
        
        X_train, X_test, y_train, y_test = split_data(X_train_data, y, test_size=0.2, random_state=42)
        
        best_params, study, best_trial = create_optuna_rf_optimizer(
            X_train, y_train, X_test, y_test,
            n_trials=n_trials,
            cv_folds=5
        )
        
        # Train final model
        final_model = RandomForestClassifier(**best_params, n_jobs=-1, random_state=42)
        final_model.fit(X_train, y_train)
        
        # Get test metrics
        y_pred = final_model.predict(X_test)
        test_metrics = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1': f1_score(y_test, y_pred),
        }
        
        results[target_name] = {
            'best_params': best_params,
            'best_score': best_trial.value,
            'test_metrics': test_metrics,
            'model': final_model,
            'study': study
        }
        
        print(f"Best Accuracy (CV): {best_trial.value:.4f}")
        print(f"Test Accuracy: {test_metrics['Accuracy']:.4f}")
    
    return results

# Uncomment below to optimize for both targets (this may take a while):
# targets = {"IncomeInvestment": y_income, "AccumulationInvestment": y_accum}
# all_results = optimize_rf_for_all_targets(X_engineered, targets, "Engineered", n_trials=50)

# Save results summary
print("\n" + "="*80)
print("OPTUNA OPTIMIZATION SUMMARY")
print("="*80)
print("\nTo apply optimized hyperparameters in your pipeline:")
print("\n```python")
print("optimized_rf = RandomForestClassifier(")
for key, value in best_params.items():
    if isinstance(value, str):
        print(f"    {key}='{value}',")
    else:
        print(f"    {key}={value},")
print("    n_jobs=-1,")
print("    random_state=42")
print(")")
print("\nmodel = create_pipeline(optimized_rf, scaler=False)")
print("```")